In [ ]:
!pip install lightgbm
!pip install xgboost
!pip install catboost
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

In [330]:
train = pd.read_csv('office_train.csv')
X = train.drop('OfficeCategory', axis=1)
y = train['OfficeCategory']

print("Dataset loaded successfully!")
print(f"Shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts().sort_index()}")

Dataset loaded successfully!
Shape: (35000, 79)
Target distribution:
OfficeCategory
0    6675
1    7314
2    6906
3    7013
4    7092
Name: count, dtype: int64


In [331]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from datetime import datetime

def _add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    # Get current year for age calculations
    NOW = datetime.now().year

    if {'BuildingGrade', 'OfficeSpace'}.issubset(df.columns):
        df['Quality_Size'] = df['BuildingGrade'] * df['OfficeSpace']
    
    if 'OfficeSpace' in df.columns:
        df['OfficeSpace_squared'] = np.round(df['OfficeSpace'] ** 2)
        df['OfficeSpace_sqrt'] = np.sqrt(df['OfficeSpace'] + 1)
    
    if 'BuildingGrade' in df.columns:
        df['BuildingGrade_squared'] = df['BuildingGrade'] ** 2
    
    if 'PlotSize' in df.columns:
        df['PlotSize_sqrt'] = np.sqrt(df['PlotSize'] + 1)
    
    if {'OfficeSpace', 'PlotSize'}.issubset(df.columns):
        df['Space_Plot_Ratio'] = np.round(df['OfficeSpace'] / (df['PlotSize'] + 1), 8)
        df['Plot_Space_Ratio'] = np.round(df['PlotSize'] / (df['OfficeSpace'] + 1), 8)
    
    if {'Restrooms', 'MeetingRooms'}.issubset(df.columns):
        df['Restroom_Meeting_Ratio'] = np.round(df['Restrooms'] / (df['MeetingRooms'] + 1), 4)
        df['Meeting_Restroom_Ratio'] = np.round(df['MeetingRooms'] / (df['Restrooms'] + 1), 4)
    
    if {'OfficeSpace', 'Restrooms'}.issubset(df.columns):
        df['Space_Per_Restroom'] = np.round(df['OfficeSpace'] / (df['Restrooms'] + 1), 2)
    
    if {'OfficeSpace', 'MeetingRooms'}.issubset(df.columns):
        df['Space_Per_MeetingRoom'] = np.round(df['OfficeSpace'] / (df['MeetingRooms'] + 1), 2)
    
    needed = {'OfficeSpace', 'BasementArea', 'ParkingArea'}
    
    if needed.issubset(df.columns):
        df['TotalArea'] = np.round((df['OfficeSpace'] + df['BasementArea'] + df['ParkingArea']), 2)
        df['OfficeSpace_Pct'] = np.round(df['OfficeSpace'] / (df['TotalArea'] + 1), 4)
        df['BasementArea_Pct'] = np.round(df['BasementArea'] / (df['TotalArea'] + 1), 4)
        df['ParkingArea_Pct'] = np.round(df['ParkingArea'] / (df['TotalArea'] + 1), 4)
    
    if {'MeetingRooms', 'TotalArea'}.issubset(df.columns):
        df['MeetingRooms_Per_TotalArea'] = np.round(df['MeetingRooms'] / (df['TotalArea'] + 1), 6)
    
    if {'YearListed', 'RenovationYear', 'ConstructionYear'}.issubset(df.columns):
        df['Years_Since_Construction'] = df['YearListed'] - df['ConstructionYear']
        df['Years_Since_Renovation'] = df['YearListed'] - df['RenovationYear']
        df['Recent_Renovation'] = (df['RenovationYear'] >= df['YearListed'] - 5).astype(int)
        df['Age_At_Listing'] = df['YearListed'] - df['ConstructionYear']
        df['Renovation_Frequency'] = np.round(df['Years_Since_Renovation'] / (df['Age_At_Listing'] + 1), 4)
    
    if {'BasementArea', 'TotalArea'}.issubset(df.columns):
        df['Basement_TotalArea_Ratio'] = df['BasementArea'] / (df['TotalArea'] + 1)
    
    if {'FinishedBasementArea1', 'FinishedBasementArea2', 'BasementArea'}.issubset(df.columns):
        finished_basement_total = df['FinishedBasementArea1'] + df['FinishedBasementArea2']
        df['FinishedBasement_Ratio'] = np.round(finished_basement_total / (df['BasementArea'] + 1), 8)
        df['UnfinishedBasement_Ratio'] = np.round((df['BasementArea'] - finished_basement_total) / (df['BasementArea'] + 1), 8)
    
    if {'BuildingGrade', 'Years_Since_Renovation'}.issubset(df.columns):
        df['Quality_Age_Interaction'] = df['BuildingGrade'] * df['Years_Since_Renovation']
    
    if {'OfficeSpace', 'Restrooms', 'MeetingRooms'}.issubset(df.columns):
        df['Total_Facilities'] = df['Restrooms'] + df['MeetingRooms']
        df['Space_Per_Facility'] = np.round(df['OfficeSpace'] / (df['Total_Facilities'] + 1), 2)
    
    # Age & Lifecycle Features
    # 1. Effective age (post-renovation)
    if {'YearBuilt', 'YearRenovated'}.issubset(df.columns):
        df['effective_age'] = (NOW - df[['YearBuilt', 'YearRenovated']].max(axis=1)).clip(lower=0)
    elif 'YearBuilt' in df.columns:
        df['effective_age'] = (NOW - df['YearBuilt']).clip(lower=0)
    
    # 2. Recent refresh flag (renovated in last 5 years)
    if 'YearRenovated' in df.columns:
        df['recent_refresh_5y'] = ((NOW - df['YearRenovated']) <= 5).astype(int)
    
    # 3. Vintage bucket (non-linear age bins)
    if 'YearBuilt' in df.columns:
        age = (NOW - df['YearBuilt']).clip(lower=0)
        df['age_bin'] = pd.cut(age, bins=[-1, 5, 15, 30, 10_000], labels=['0-5', '6-15', '16-30', '31+'])
        
    df = df.drop(columns=['AlleyAccess', 'UtilitiesAvailable', 'Proximity2', 'ExteriorCondition', 'LowQualityArea', 'RecreationQuality', 'RecreationArea', 'HeatingType', 'Perimeter', 'MiscellaneousValue', 'ParkingQuality', '3SsnPorch', 'MiscellaneousFeature'], errors='ignore')
    return df

def target_encode_categorical(X_train, y_train, X_test, categorical_features, smoothing=1.0):
    X_train_encoded = X_train.copy()
    X_test_encoded = X_test.copy() if X_test is not None else None
    temp_df = pd.DataFrame()
    for col in categorical_features:
        if col in X_train.columns:
            temp_df[col] = X_train[col]
    temp_df['_target'] = y_train.values
    for col in categorical_features:
        if col not in X_train.columns:
            continue
        global_mean = y_train.mean()
        category_stats = temp_df.groupby(col)['_target'].agg(['mean', 'count'])
        category_means = category_stats['mean']
        category_counts = category_stats['count']
        smoothing_factor = 1 / (1 + np.exp(-(category_counts - smoothing)))
        category_encoded = smoothing_factor * category_means + (1 - smoothing_factor) * global_mean
        X_train_encoded[col] = X_train[col].map(category_encoded).fillna(global_mean)
        if X_test_encoded is not None and col in X_test_encoded.columns:
            X_test_encoded[col] = X_test[col].map(category_encoded).fillna(global_mean)
    return X_train_encoded, X_test_encoded

def preprocess(X_train: pd.DataFrame, X_test: pd.DataFrame = None, y_train: pd.Series = None, use_target_encoding: bool = True):
    X_train = X_train.copy()
    X_test = X_test.copy() if X_test is not None else None
    X_train = _add_engineered_features(X_train)
    if X_test is not None:
        X_test = _add_engineered_features(X_test)
    numeric_features = X_train.select_dtypes(include=[np.number]).columns
    categorical_features = X_train.select_dtypes(include=['object', 'category']).columns
    print(f"Numeric features: {len(numeric_features)}")
    print(f"Categorical features: {len(categorical_features)}")
    for col in numeric_features:
        median_val = X_train[col].median()
        X_train[col] = X_train[col].fillna(median_val)
        if X_test is not None and col in X_test.columns:
            X_test[col] = X_test[col].fillna(median_val)
    for col in categorical_features:
        mode_series = X_train[col].mode()
        mode_val = mode_series.iloc[0] if len(mode_series) > 0 else 'Missing'
        X_train[col] = X_train[col].fillna(mode_val)
        if X_test is not None and col in X_test.columns:
            X_test[col] = X_test[col].fillna(mode_val)
    if use_target_encoding and y_train is not None and len(categorical_features) > 0:
        print("Using target encoding for categorical features...")
        X_train, X_test = target_encode_categorical(X_train, y_train, X_test, categorical_features)
        for col in categorical_features:
            if col in X_train.columns:
                X_train[col] = pd.to_numeric(X_train[col], errors='coerce')
            if X_test is not None and col in X_test.columns:
                X_test[col] = pd.to_numeric(X_test[col], errors='coerce')
    else:
        print("Using label encoding for categorical features...")
        for col in categorical_features:
            le = LabelEncoder()
            if X_test is not None and col in X_test.columns:
                combined = pd.concat([X_train[col].astype(str), X_test[col].astype(str)], axis=0)
                le.fit(combined)
                X_train[col] = le.transform(X_train[col].astype(str))
                X_test[col] = le.transform(X_test[col].astype(str))
            else:
                X_train[col] = le.fit_transform(X_train[col].astype(str))
    X_train = X_train.fillna(0)
    if X_test is not None:
        X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
        return X_train, X_test
    return X_train

X_processed = preprocess(X, y_train=y, use_target_encoding=True)
print(f"\nAfter preprocessing:")
print(f"Shape: {X_processed.shape}")
print(f"Missing values: {X_processed.isnull().sum().sum()}")
X_processed.head(20)
X_processed.to_csv("clean.csv")


Numeric features: 59
Categorical features: 34
Using target encoding for categorical features...

After preprocessing:
Shape: (35000, 93)
Missing values: 0


In [332]:
X_train, X_val, y_train, y_val = train_test_split(X_processed, y, test_size=0.2, random_state=42, stratify=y)
print(f"\nTrain set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")


Train set: (28000, 93)
Validation set: (7000, 93)


In [333]:
print("\n" + "="*50)
print("TRAINING OPTIMIZED CATBOOST MODEL")
print("="*50)

model = CatBoostClassifier(
    loss_function='MultiClass',
    iterations=1000,
    depth=10,
    learning_rate=0.03,
    l2_leaf_reg=3,
    border_count=128,
    eval_metric='Accuracy',
    verbose=100,
    early_stopping_rounds=100,
    random_seed=42,
    task_type='CPU',
    use_best_model=True
)

model.fit(X_train, y_train, eval_set=(X_val, y_val), plot=False)
print(f"\nBest iteration: {model.get_best_iteration()}")
print(f"Best score: {model.get_best_score()}")


TRAINING OPTIMIZED CATBOOST MODEL
0:	learn: 0.6365000	test: 0.6381429	best: 0.6381429 (0)	total: 106ms	remaining: 1m 45s
100:	learn: 0.8286071	test: 0.8191429	best: 0.8191429 (100)	total: 3.94s	remaining: 35.1s
200:	learn: 0.8620714	test: 0.8422857	best: 0.8424286 (195)	total: 7.47s	remaining: 29.7s
300:	learn: 0.8844643	test: 0.8484286	best: 0.8484286 (300)	total: 11.4s	remaining: 26.6s
400:	learn: 0.9016071	test: 0.8551429	best: 0.8551429 (400)	total: 14.9s	remaining: 22.2s
500:	learn: 0.9131071	test: 0.8584286	best: 0.8585714 (493)	total: 18.4s	remaining: 18.3s
600:	learn: 0.9228929	test: 0.8610000	best: 0.8618571 (567)	total: 21.8s	remaining: 14.5s
700:	learn: 0.9309643	test: 0.8621429	best: 0.8635714 (677)	total: 25.1s	remaining: 10.7s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.8635714286
bestIteration = 677

Shrink model to first 678 iterations.

Best iteration: 677
Best score: {'learn': {'Accuracy': 0.9368928571428572, 'MultiClass': 0.29930054728228606

In [334]:
y_train_pred = model.predict(X_train)
y_val_pred = model.predict(X_val)
train_acc = accuracy_score(y_train, y_train_pred)
val_acc = accuracy_score(y_val, y_val_pred)
print(f"\nTrain Accuracy: {train_acc*100:.2f}%")
print(f"Validation Accuracy: {val_acc*100:.2f}%")
print("\nClassification Report (Validation):")
print(classification_report(y_val, y_val_pred))


Train Accuracy: 92.90%
Validation Accuracy: 86.36%

Classification Report (Validation):
              precision    recall  f1-score   support

           0       0.87      0.88      0.87      1335
           1       0.79      0.80      0.79      1463
           2       0.82      0.83      0.82      1381
           3       0.88      0.87      0.88      1403
           4       0.96      0.94      0.95      1418

    accuracy                           0.86      7000
   macro avg       0.86      0.86      0.86      7000
weighted avg       0.86      0.86      0.86      7000



In [335]:
print("\n" + "="*50)
print("TRAINING ENSEMBLE MODEL")
print("="*50)

X_train_clean = X_train.copy()
X_val_clean = X_val.copy()
for col in X_train_clean.columns:
    X_train_clean[col] = pd.to_numeric(X_train_clean[col], errors='coerce')
    X_val_clean[col] = pd.to_numeric(X_val_clean[col], errors='coerce')
X_train_clean = X_train_clean.fillna(0)
X_val_clean = X_val_clean.fillna(0)
X_train_clean = X_train_clean.replace([np.inf, -np.inf], 0)
X_val_clean = X_val_clean.replace([np.inf, -np.inf], 0)
y_train_clean = y_train.astype(int)
y_val_clean = y_val.astype(int)

catboost_model = model
print("\nTraining CatBoost for ensemble...")
try:
    best_iter = model.get_best_iteration()
    if best_iter is None or best_iter == 0:
        best_iter = 500
except:
    best_iter = 500

catboost_ensemble = CatBoostClassifier(
    loss_function='MultiClass',
    iterations=best_iter,
    depth=10,
    learning_rate=0.03,
    l2_leaf_reg=3,
    border_count=128,
    eval_metric='Accuracy',
    verbose=False,
    random_seed=42,
    task_type='CPU',
    use_best_model=False
)
catboost_ensemble.fit(X_train_clean, y_train_clean)

print("\nTraining XGBoost...")
xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss',
    early_stopping_rounds=50,
    n_jobs=-1,
    tree_method='hist'
)
xgb_model.fit(X_train_clean, y_train_clean, eval_set=[(X_val_clean, y_val_clean)], verbose=100)

print("\nTraining LightGBM...")
lgbm_model = LGBMClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1,
    early_stopping_round=50
)
lgbm_model.fit(X_train_clean, y_train_clean, eval_set=[(X_val_clean, y_val_clean)], callbacks=[early_stopping(50), log_evaluation(100)])

try:
    lgbm_best_iter = lgbm_model.best_iteration_
    if lgbm_best_iter is None or lgbm_best_iter == 0:
        lgbm_best_iter = 500
except:
    lgbm_best_iter = 500

xgb_best_iter = min(lgbm_best_iter, 500)

print("\nCreating ensemble models...")
xgb_ensemble = XGBClassifier(
    n_estimators=xgb_best_iter,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss',
    n_jobs=-1,
    tree_method='hist'
)

lgbm_ensemble = LGBMClassifier(
    n_estimators=lgbm_best_iter,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)

ensemble_model = VotingClassifier(
    estimators=[('catboost', catboost_ensemble), ('xgb', xgb_ensemble), ('lgbm', lgbm_ensemble)],
    voting='soft',
    weights=[1.2, 1.0, 1.0]
)

print("\n" + "="*50)
print("INDIVIDUAL MODEL PERFORMANCE")
print("="*50)

models_dict = {
    'CatBoost': (catboost_model, X_val, y_val),
    'XGBoost': (xgb_model, X_val_clean, y_val_clean),
    'LightGBM': (lgbm_model, X_val_clean, y_val_clean)
}

for name, (m, X_eval, y_eval) in models_dict.items():
    pred = m.predict(X_eval)
    acc = accuracy_score(y_eval, pred)
    print(f"{name} Validation Accuracy: {acc*100:.2f}%")

print("\n" + "="*50)
print("ENSEMBLE MODEL PERFORMANCE")
print("="*50)

ensemble_model.fit(X_train_clean, y_train_clean)
ensemble_pred = ensemble_model.predict(X_val_clean)
ensemble_acc = accuracy_score(y_val_clean, ensemble_pred)
print(f"Ensemble Validation Accuracy: {ensemble_acc*100:.2f}%")

catboost_acc = accuracy_score(y_val, catboost_model.predict(X_val))

if ensemble_acc > catboost_acc:
    print(f"\n✓ Ensemble model performs better! ({ensemble_acc*100:.2f}% vs {catboost_acc*100:.2f}%)")
    print("Using ensemble for predictions...")
    final_model = ensemble_model
    use_cleaned_data = True
else:
    print(f"\n✓ CatBoost model performs better. ({catboost_acc*100:.2f}% vs {ensemble_acc*100:.2f}%)")
    print("Using CatBoost for predictions...")
    final_model = catboost_model
    use_cleaned_data = False



TRAINING ENSEMBLE MODEL

Training CatBoost for ensemble...

Training XGBoost...
[0]	validation_0-mlogloss:1.56964
[100]	validation_0-mlogloss:0.56132
[200]	validation_0-mlogloss:0.45364
[300]	validation_0-mlogloss:0.42686
[400]	validation_0-mlogloss:0.41762
[499]	validation_0-mlogloss:0.41448

Training LightGBM...
Training until validation scores don't improve for 50 rounds
[100]	valid_0's multi_logloss: 0.510576
[200]	valid_0's multi_logloss: 0.440681
[300]	valid_0's multi_logloss: 0.424978
[400]	valid_0's multi_logloss: 0.417652
[500]	valid_0's multi_logloss: 0.414441
Did not meet early stopping. Best iteration is:
[500]	valid_0's multi_logloss: 0.414441

Creating ensemble models...

INDIVIDUAL MODEL PERFORMANCE
CatBoost Validation Accuracy: 86.36%
XGBoost Validation Accuracy: 86.07%
LightGBM Validation Accuracy: 86.11%

ENSEMBLE MODEL PERFORMANCE
Ensemble Validation Accuracy: 86.51%

✓ Ensemble model performs better! (86.51% vs 86.36%)
Using ensemble for predictions...


In [336]:
test = pd.read_csv('office_test.csv')
X_test_processed = preprocess(X, test, y_train=y, use_target_encoding=True)[1]

try:
    prediction_model = final_model
    needs_cleaned_data = use_cleaned_data
    print("Using ensemble model for predictions...")
except NameError:
    prediction_model = model
    needs_cleaned_data = False
    print("Using single CatBoost model for predictions...")

if needs_cleaned_data:
    X_test_clean = X_test_processed.copy()
    for col in X_test_clean.columns:
        X_test_clean[col] = pd.to_numeric(X_test_clean[col], errors='coerce')
    X_test_clean = X_test_clean.fillna(0).replace([np.inf, -np.inf], 0)
    X_test_for_prediction = X_test_clean
else:
    X_test_for_prediction = X_test_processed

test_predictions = prediction_model.predict(X_test_for_prediction)
if len(test_predictions.shape) > 1:
    test_predictions = test_predictions.flatten()

submission = pd.DataFrame({
    'Id': range(len(test_predictions)),
    'OfficeCategory': test_predictions
})
submission.to_csv('submission.csv', index=False)

print("Predictions saved to submission.csv")
print(f"Total predictions: {len(test_predictions)}")
print(f"Prediction distribution:\n{pd.Series(test_predictions).value_counts().sort_index()}")

Numeric features: 59
Categorical features: 34
Using target encoding for categorical features...
Using ensemble model for predictions...
Predictions saved to submission.csv
Total predictions: 15000
Prediction distribution:
0    2885
1    3210
2    2958
3    2987
4    2960
Name: count, dtype: int64
